In [8]:
# Supported values are: 'code_interpreter', 'function', and 'file_search'."라는 오류가 발생합니다.
# retrieval를 file_search로 변경한 예제입니다

from openai import OpenAI
import os

# os.environ["OPENAI_API_KEY"] = "sk-"
# client = OpenAI()

In [9]:
from google.colab import userdata
openai_key = userdata.get('openai_key')

client = OpenAI(
    api_key=openai_key)

In [10]:
import json
def show_json(obj):
  display(json.loads(obj.model_dump_json()))
# model_dump_json()는 객체의 데이터를 JSON 문자열로 반환
# json.loads() 파이썬 딕셔너리로 변환

assistant = client.beta.assistants.create(
  name="smartfarm",
  instructions="당신은 스마트팜에 대한 전문가입니다. pdf 파일에 기반하여 답변해주세요.",
  model="gpt-4o",
  tools=[{"type": "file_search"}]  # 파일 검색 기능을 활용
)
show_json(assistant)

{'id': 'asst_Iusve6mekEAcqyFjXJR5iDXl',
 'created_at': 1737307379,
 'description': None,
 'instructions': '당신은 스마트팜에 대한 전문가입니다. pdf 파일에 기반하여 답변해주세요.',
 'metadata': {},
 'model': 'gpt-4o',
 'name': 'smartfarm',
 'object': 'assistant',
 'tools': [{'type': 'file_search',
   'file_search': {'max_num_results': None,
    'ranking_options': {'score_threshold': 0.0,
     'ranker': 'default_2024_08_21'}}}],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': None,
  'file_search': {'vector_store_ids': []}},
 'top_p': 1.0}

In [12]:
file = "스마트팜.pdf"

In [13]:
vector_store = client.beta.vector_stores.create(
    name="file_store"
)

# 업로드할 파일 PDF 준비
file_paths = [file]
file_streams = [open(path, "rb") for path in file_paths]

# 파일을 업로드하고, vector store에 추가
file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
  vector_store_id = vector_store.id, files=file_streams,
)

# 파일의 업로드 상태 확인
print(file_batch.status)
print(file_batch.file_counts)

show_json(vector_store)

completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)


{'id': 'vs_tqpopYqAAzzuVqMZAxsMsxdk',
 'created_at': 1737307591,
 'file_counts': {'cancelled': 0,
  'completed': 0,
  'failed': 0,
  'in_progress': 0,
  'total': 0},
 'last_active_at': 1737307591,
 'metadata': {},
 'name': 'file_store',
 'object': 'vector_store',
 'status': 'completed',
 'usage_bytes': 0,
 'expires_after': None,
 'expires_at': None}

In [14]:
# 생성한 vector store를 참조할 수 있도록 업데이트
assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)
show_json(assistant)

{'id': 'asst_Iusve6mekEAcqyFjXJR5iDXl',
 'created_at': 1737307379,
 'description': None,
 'instructions': '당신은 스마트팜에 대한 전문가입니다. pdf 파일에 기반하여 답변해주세요.',
 'metadata': {},
 'model': 'gpt-4o',
 'name': 'smartfarm',
 'object': 'assistant',
 'tools': [{'type': 'file_search',
   'file_search': {'max_num_results': None,
    'ranking_options': {'score_threshold': 0.0,
     'ranker': 'default_2024_08_21'}}}],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': None,
  'file_search': {'vector_store_ids': ['vs_tqpopYqAAzzuVqMZAxsMsxdk']}},
 'top_p': 1.0}

In [15]:
# thread, message, run 생성 및 실행
thread = client.beta.threads.create()

message = client.beta.threads.messages.create(
    thread_id = thread.id,
    role = "user",
    content= "스마트팜이란?",
)

run = client.beta.threads.runs.create_and_poll(
    thread_id = thread.id,
    assistant_id = assistant.id,
)
show_json(run)

{'id': 'run_fbcEpEt5KE3TkmUELezfN4Td',
 'assistant_id': 'asst_Iusve6mekEAcqyFjXJR5iDXl',
 'cancelled_at': None,
 'completed_at': 1737307613,
 'created_at': 1737307608,
 'expires_at': None,
 'failed_at': None,
 'incomplete_details': None,
 'instructions': '당신은 스마트팜에 대한 전문가입니다. pdf 파일에 기반하여 답변해주세요.',
 'last_error': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'metadata': {},
 'model': 'gpt-4o',
 'object': 'thread.run',
 'parallel_tool_calls': True,
 'required_action': None,
 'response_format': 'auto',
 'started_at': 1737307609,
 'status': 'completed',
 'thread_id': 'thread_x7EkB07OUmruCcshAxM4mfBU',
 'tool_choice': 'auto',
 'tools': [{'type': 'file_search',
   'file_search': {'max_num_results': None,
    'ranking_options': {'score_threshold': 0.0,
     'ranker': 'default_2024_08_21'}}}],
 'truncation_strategy': {'type': 'auto', 'last_messages': None},
 'usage': {'completion_tokens': 100,
  'prompt_tokens': 13350,
  'total_tokens': 13450,
  'prompt_token_details': {'

In [16]:
# GPT-4o 답변 출력
if run.status == 'completed':
  messages = client.beta.threads.messages.list(
    thread_id=thread.id
  )
  show_json(messages)
  #print_message(thread.id)
else:
  print(run.status)

{'data': [{'id': 'msg_JKCcvMvV29jKDTUlGF5Jsotq',
   'assistant_id': 'asst_Iusve6mekEAcqyFjXJR5iDXl',
   'attachments': [],
   'completed_at': None,
   'content': [{'text': {'annotations': [{'end_index': 125,
        'file_citation': {'file_id': 'file-TDqxkHf26aYvGUAaS3cbcn'},
        'start_index': 111,
        'text': '【4:1†스마트팜.pdf】',
        'type': 'file_citation'}],
      'value': '스마트팜이란 광의적인 의미에서 정보통신기술(ICT)을 농업의 생산, 가공, 유통 및 소비 전반에 접목하여 원격에서 자동으로 작물의 생육 환경을 관리하고, 생산 효율성을 높일 수 있는 농장을 의미합니다【4:1†스마트팜.pdf】.'},
     'type': 'text'}],
   'created_at': 1737307612,
   'incomplete_at': None,
   'incomplete_details': None,
   'metadata': {},
   'object': 'thread.message',
   'role': 'assistant',
   'run_id': 'run_fbcEpEt5KE3TkmUELezfN4Td',
   'status': None,
   'thread_id': 'thread_x7EkB07OUmruCcshAxM4mfBU'},
  {'id': 'msg_7kWw4Wz3VLOIipcEVAfqvSY8',
   'assistant_id': None,
   'attachments': [],
   'completed_at': None,
   'content': [{'text': {'annotations': [], 'value': '스마트팜이란?'},
   